In [2]:
# ============================================================
# SIMULATED A/B TEST
# Guided First-Session Activation Flow
# ============================================================

import math

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportion_effectsize,
    proportions_ztest,
    confint_proportions_2indep
)


# ------------------------------------------------------------
# 1. SAMPLE SIZE PLANNING
# ------------------------------------------------------------

baseline = 0.158
target = 0.173

alpha = 0.05
power = 0.80

practical_threshold = 0.015  # +1.5 percentage points

effect_size = proportion_effectsize(target, baseline)

power_analysis = NormalIndPower()

n_per_group = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,
    alternative="two-sided"
)

required_per_group = math.ceil(n_per_group)
required_total = required_per_group * 2

print("Required sample per group:", required_per_group)
print("Required total sample:", required_total)


# ------------------------------------------------------------
# 2. SIMULATED EXPERIMENT RESULT
# ------------------------------------------------------------

control_users = 10000
control_retained = 1580

treatment_users = 10000
treatment_retained = 1750

control_rate = control_retained / control_users
treatment_rate = treatment_retained / treatment_users

absolute_uplift = treatment_rate - control_rate
relative_uplift = absolute_uplift / control_rate


# ------------------------------------------------------------
# 3. TWO-PROPORTION Z-TEST
# ------------------------------------------------------------

successes = [
    treatment_retained,
    control_retained
]

observations = [
    treatment_users,
    control_users
]

z_stat, p_value = proportions_ztest(
    successes,
    observations,
    alternative="two-sided"
)


# ------------------------------------------------------------
# 4. 95% CONFIDENCE INTERVAL
# Treatment - Control
# ------------------------------------------------------------

ci_low, ci_high = confint_proportions_2indep(
    count1=treatment_retained,
    nobs1=treatment_users,
    count2=control_retained,
    nobs2=control_users,
    method="wald"
)


# ------------------------------------------------------------
# 5. PRODUCT DECISION
# ------------------------------------------------------------

if (
    p_value < alpha
    and absolute_uplift >= practical_threshold
    and ci_low >= practical_threshold
):
    decision = "SHIP"

elif p_value < alpha and absolute_uplift > 0:
    decision = "ITERATE"

else:
    decision = "REJECT"


# ------------------------------------------------------------
# 6. OUTPUT
# ------------------------------------------------------------

print("\n--- SIMULATED EXPERIMENT ---")

print(f"Control D1:   {control_rate:.2%}")
print(f"Treatment D1: {treatment_rate:.2%}")

print(
    f"Absolute uplift: "
    f"{absolute_uplift * 100:.2f} percentage points"
)

print(
    f"Relative uplift: "
    f"{relative_uplift * 100:.2f}%"
)

print(f"Z-statistic: {z_stat:.3f}")
print(f"P-value: {p_value:.4f}")

print(
    "95% CI for uplift: "
    f"{ci_low * 100:.2f} pp "
    f"to {ci_high * 100:.2f} pp"
)

print(
    f"Practical threshold: "
    f"{practical_threshold * 100:.2f} pp"
)

print(f"Decision: {decision}")


Required sample per group: 9632
Required total sample: 19264

--- SIMULATED EXPERIMENT ---
Control D1:   15.80%
Treatment D1: 17.50%
Absolute uplift: 1.70 percentage points
Relative uplift: 10.76%
Z-statistic: 3.227
P-value: 0.0013
95% CI for uplift: 0.67 pp to 2.73 pp
Practical threshold: 1.50 pp
Decision: ITERATE


## Interpretation

The simulated treatment increased D1 retention from 15.8% to 17.5%, a +1.7 percentage-point uplift.

The result is statistically significant, but the 95% confidence interval still includes effects below the predefined +1.5 percentage-point practical threshold.

**Decision: Iterate**

The treatment is promising, but I would gather more evidence or test an improved version before recommending a full rollout.